# Stain normalization on full dataset

Applies the chosen strategy (reference + Macenko vs Reinhard by blue_dom_pct) to the **entire** train set. Computes **per-image** and **dataset-level** metrics to assess success.

## 1. Load data and reference

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "pcam-master"))
DATA_DIR = os.path.join(PROJECT_ROOT, "pcam_data")
from keras_pcam.dataset.pcam import load_data

(train_x, train_y, _), _, _ = load_data(data_dir=DATA_DIR)
n_total = len(train_x)
print("Train size:", n_total)

In [ ]:
ref_path = os.path.join(PROJECT_ROOT, "experiments", "stain_reference", "stain_reference.json")
with open(ref_path) as f:
    ref_config = json.load(f)
ref_train_idx = ref_config["reference_train_index"]
ref_patch = np.asarray(train_x[ref_train_idx])
ref_patch = np.clip(ref_patch.astype(np.float64) / 255.0, 0, 1) if ref_patch.max() > 1 else np.clip(ref_patch.astype(np.float64), 0, 1)
ref_uint8 = (ref_patch * 255).astype(np.uint8)
ref_mean_r = float(np.mean(ref_patch[:,:,0]))
ref_mean_g = float(np.mean(ref_patch[:,:,1]))
ref_mean_b = float(np.mean(ref_patch[:,:,2]))
print("Reference index:", ref_train_idx, "| mean RGB:", round(ref_mean_r,4), round(ref_mean_g,4), round(ref_mean_b,4))

Reference index: 76889 | mean RGB: 0.6137 0.3966 0.6954


## 2. Fit normalizers and compute blue_dom threshold

In [ ]:
from staintools import StainNormalizer, ReinhardColorNormalizer
from staintools.preprocessing.luminosity_standardizer import LuminosityStandardizer

macenko = StainNormalizer(method='macenko')
ref_fit = LuminosityStandardizer.standardize(ref_uint8.copy())
macenko.fit(ref_fit)
reinhard = ReinhardColorNormalizer()
reinhard.fit(ref_uint8)
print("Macenko and Reinhard fitted.")

Macenko and Reinhard fitted.


In [ ]:
N_SAMPLE = min(30000, n_total)
blue_dom_list = []
for i in tqdm(range(N_SAMPLE), desc="blue_dom"):
    p = np.asarray(train_x[i])
    p = np.clip(p.astype(np.float64) / 255.0, 0, 1) if p.max() > 1 else np.clip(p.astype(np.float64), 0, 1)
    blue_dom_list.append((p[:,:,2] > p[:,:,0]).mean())
blue_dom_threshold = np.percentile(blue_dom_list, 25)
print("blue_dom_pct threshold (25th pct):", round(blue_dom_threshold, 4))

blue_dom: 100%|██████████| 30000/30000 [00:13<00:00, 2296.66it/s]

blue_dom_pct threshold (25th pct): 0.0486


## 3. Helpers: to_uint8, contrast_stretch, per-image metrics

In [ ]:
def to_uint8(p):
    if p.max() > 1.0:
        return np.clip(p, 0, 255).astype(np.uint8)
    return (np.clip(p, 0, 1) * 255).astype(np.uint8)

def contrast_stretch(img, pl=2, ph=98):
    lo, hi = np.percentile(img, (pl, ph))
    if hi <= lo:
        return img.astype(np.uint8)
    out = (np.clip(img.astype(np.float32), lo, hi) - lo) / (hi - lo) * 255
    return np.clip(out, 0, 255).astype(np.uint8)

def per_image_metrics(img_uint8):
    p = img_uint8.astype(np.float64) / 255.0
    r, g, b = p[:,:,0], p[:,:,1], p[:,:,2]
    mx, mn = p.max(axis=2), p.min(axis=2)
    sat = np.where(mx > 1e-8, (mx - mn) / np.maximum(mx, 1e-8), 0.0)
    return float(np.mean(r)), float(np.mean(g)), float(np.mean(b)), float(np.std(r)), float(np.std(g)), float(np.std(b)), float(np.mean(sat))

## 4. Normalize entire dataset and collect per-image metrics

In [ ]:
before_r, before_g, before_b = [], [], []
after_r, after_g, after_b = [], [], []
after_std_r, after_std_g, after_std_b = [], [], []
after_sat = []
blue_dom_used, used_reinhard = [], []
fallback_no_stain_indices = []
np.random.seed(42)
N_CACHE = 400
cache_indices = set(np.random.choice(n_total, min(N_CACHE, n_total), replace=False).tolist())
cached_orig, cached_norm = {}, {}

for i in tqdm(range(n_total), desc="Normalize"):
    p = np.asarray(train_x[i])
    p01 = np.clip(p.astype(np.float64) / 255.0, 0, 1) if p.max() > 1 else np.clip(p.astype(np.float64), 0, 1)
    bd = (p01[:,:,2] > p01[:,:,0]).mean()
    blue_dom_used.append(bd)
    use_reinhard = bd < blue_dom_threshold
    used_reinhard.append(1 if use_reinhard else 0)
    before_r.append(float(np.mean(p01[:,:,0])))
    before_g.append(float(np.mean(p01[:,:,1])))
    before_b.append(float(np.mean(p01[:,:,2])))
    pu8 = to_uint8(p)
    pstd = LuminosityStandardizer.standardize(pu8.copy())
    try:
        norm_u8 = reinhard.transform(pstd) if use_reinhard else macenko.transform(pstd)
    except Exception:
        try:
            norm_u8 = macenko.transform(pstd) if use_reinhard else reinhard.transform(pstd)
        except Exception:
            norm_u8 = pstd
            fallback_no_stain_indices.append(i)
    mr, mg, mb, sr, sg, sb, msat = per_image_metrics(norm_u8)
    after_r.append(mr)
    after_g.append(mg)
    after_b.append(mb)
    after_std_r.append(sr)
    after_std_g.append(sg)
    after_std_b.append(sb)
    after_sat.append(msat)
    if i in cache_indices:
        cached_orig[i] = pu8.copy()
        cached_norm[i] = contrast_stretch(norm_u8.copy())

before_r = np.array(before_r)
before_g = np.array(before_g)
before_b = np.array(before_b)
after_r = np.array(after_r)
after_g = np.array(after_g)
after_b = np.array(after_b)
after_std_r = np.array(after_std_r)
after_std_g = np.array(after_std_g)
after_std_b = np.array(after_std_b)
after_sat = np.array(after_sat)
blue_dom_used = np.array(blue_dom_used)
used_reinhard = np.array(used_reinhard)
n_reinhard = int(used_reinhard.sum())
n_macenko = n_total - n_reinhard
fallback_no_stain_indices = np.array(fallback_no_stain_indices)
n_fallback_no_stain = len(fallback_no_stain_indices)
print("Reinhard:", n_reinhard, "| Macenko:", n_macenko, "| Both failed (luminosity only):", n_fallback_no_stain, "| Cached:", len(cached_norm))

Normalize:   0%|          | 6/262144 [00:00<1:30:07, 48.47it/s]

Normalize:   0%|          | 25/262144 [00:00<1:19:55, 54.65it/s]C:\Users\tamer\AppData\Local\Temp\ipykernel_3460\2125694557.py:17: RuntimeWarning: invalid value encountered in divide
  sat = np.where(mx > 1e-8, (mx - mn) / mx, 0.0)
Normalize:   0%|          | 702/262144 [00:13<1:26:29, 50.38it/s]


TissueMaskException: Empty tissue mask computed

## 5. Dataset-level success metrics (quantitative)

In [ ]:
std_before_r = np.std(before_r)
std_after_r = np.std(after_r)
dist_before = np.mean(np.abs(before_r - ref_mean_r) + np.abs(before_g - ref_mean_g) + np.abs(before_b - ref_mean_b))
dist_after = np.mean(np.abs(after_r - ref_mean_r) + np.abs(after_g - ref_mean_g) + np.abs(after_b - ref_mean_b))
out_before = ((before_r < 0.2) | (before_r > 0.8) | (before_b < 0.2) | (before_b > 0.8)).mean()
out_after = ((after_r < 0.2) | (after_r > 0.8) | (after_b < 0.2) | (after_b > 0.8)).mean()

print("Std(mean_R) before / after:", round(std_before_r, 4), "/", round(std_after_r, 4))
print("Mean L1 dist to ref before / after:", round(dist_before, 4), "/", round(dist_after, 4))
print("Outlier rate (mean R or B outside [0.2,0.8]) before / after:", round(out_before, 4), "/", round(out_after, 4))
print("Mean saturation after:", round(np.mean(after_sat), 4))

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for k, (b, a, name) in enumerate(zip([before_r, before_g, before_b], [after_r, after_g, after_b], ["R", "G", "B"])):
    ax[k].hist(b, bins=50, alpha=0.5, label="before", color="gray", density=True)
    ax[k].hist(a, bins=50, alpha=0.5, label="after", color="steelblue", density=True)
    ax[k].set_title("mean_" + name)
    ax[k].legend()
plt.suptitle("Color distribution before vs after normalization")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].scatter(before_r, before_b, s=1, alpha=0.3, c="gray")
ax[0].scatter(ref_mean_r, ref_mean_b, s=120, c="red", marker="*", label="ref")
ax[0].set_xlabel("mean_R"); ax[0].set_ylabel("mean_B"); ax[0].set_title("Before")
ax[0].set_xlim(0, 1); ax[0].set_ylim(0, 1)
ax[1].scatter(after_r, after_b, s=1, alpha=0.3, c="steelblue")
ax[1].scatter(ref_mean_r, ref_mean_b, s=120, c="red", marker="*", label="ref")
ax[1].set_xlabel("mean_R"); ax[1].set_ylabel("mean_B"); ax[1].set_title("After")
ax[1].set_xlim(0, 1); ax[1].set_ylim(0, 1)
plt.suptitle("mean_R vs mean_B (red star = reference)")
plt.tight_layout()
plt.show()

In [ ]:
show = sorted(cache_indices)[:24]
fig, axes = plt.subplots(4, 12, figsize=(24, 8))
for k, i in enumerate(show):
    r, c = k // 12, k % 12
    axes[2*r, c].imshow(cached_orig[i], interpolation="bilinear")
    axes[2*r, c].set_title(str(i)); axes[2*r, c].axis("off")
    axes[2*r+1, c].imshow(cached_norm[i], interpolation="bilinear")
    axes[2*r+1, c].axis("off")
plt.suptitle("Top: original | Bottom: normalized (contrast-stretched for display)")
plt.tight_layout()
plt.show()

In [ ]:
reinhard_idx = np.where(used_reinhard == 1)[0]
macenko_idx = np.where(used_reinhard == 0)[0]
in_cache_r = [i for i in reinhard_idx if i in cache_indices]
in_cache_m = [i for i in macenko_idx if i in cache_indices]
show_r = list(in_cache_r)[:12]
show_m = list(in_cache_m)[:12]
fig, axes = plt.subplots(4, 6, figsize=(12, 8))
for k, i in enumerate(show_r + show_m):
    r, c = k // 6, k % 6
    axes[r, c].imshow(cached_norm[i], interpolation="bilinear")
    axes[r, c].set_title("R" if k < 12 else "M")
    axes[r, c].axis("off")
plt.suptitle("Reinhard (R) vs Macenko (M) normalized")
plt.tight_layout()
plt.show()

In [ ]:
for idx in [154241, 202357]:
    if idx >= n_total:
        continue
    p = np.asarray(train_x[idx])
    pu8 = to_uint8(p)
    pstd = LuminosityStandardizer.standardize(pu8.copy())
    p01 = np.clip(p.astype(np.float64)/255.0, 0, 1) if p.max() > 1 else np.clip(p.astype(np.float64), 0, 1)
    bd = (p01[:,:,2] > p01[:,:,0]).mean()
    nr = reinhard.transform(pstd) if bd < blue_dom_threshold else macenko.transform(pstd)
    fig, ax = plt.subplots(1, 2, figsize=(4, 2))
    ax[0].imshow(pu8, interpolation="bilinear"); ax[0].set_title("Orig " + str(idx)); ax[0].axis("off")
    ax[1].imshow(contrast_stretch(nr), interpolation="bilinear"); ax[1].set_title("Norm"); ax[1].axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6, 3))
plt.hist(after_sat, bins=60, color="steelblue", edgecolor="white", alpha=0.8)
plt.xlabel("Mean saturation (per image, after norm)")
plt.ylabel("Count")
plt.title("Per-image saturation after normalization")
plt.tight_layout(); plt.show()

## 7. Save summary and per-image metrics

In [ ]:
out_dir = os.path.join(PROJECT_ROOT, "experiments", "stain_normalize_full")
os.makedirs(out_dir, exist_ok=True)
summary = {
    "n_processed": n_total, "n_reinhard": n_reinhard, "n_macenko": n_macenko,
    "n_fallback_no_stain": n_fallback_no_stain,
    "blue_dom_threshold": float(blue_dom_threshold), "ref_train_idx": ref_train_idx,
    "std_mean_r_before": float(std_before_r), "std_mean_r_after": float(std_after_r),
    "dist_to_ref_before": float(dist_before), "dist_to_ref_after": float(dist_after),
    "outlier_rate_before": float(out_before), "outlier_rate_after": float(out_after),
    "mean_saturation_after": float(np.mean(after_sat)),
}
with open(os.path.join(out_dir, "normalize_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
np.savez_compressed(os.path.join(out_dir, "per_image_metrics.npz"),
    before_r=before_r, before_g=before_g, before_b=before_b,
    after_r=after_r, after_g=after_g, after_b=after_b,
    after_std_r=after_std_r, after_std_g=after_std_g, after_std_b=after_std_b,
    after_sat=after_sat, blue_dom_pct=blue_dom_used, used_reinhard=used_reinhard)
np.save(os.path.join(out_dir, "fallback_no_stain_indices.npy"), fallback_no_stain_indices)
print("Saved", out_dir, "(summary, per_image_metrics.npz, fallback_no_stain_indices.npy)")